<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1604.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q triton

In [ ]:
import torch
import random
from typing import Optional
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)

In [ ]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    sample_size: int = 10
    test_size_ratio: float = 0.2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1


    def __post_init__(self):
        random.seed(self.seed)


    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = self.sample_size - num_with

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)

        idx_train_with_oracle, idx_test_with_oracle = train_test_split(
            indices_with,
            test_size=self.test_size_ratio,
            shuffle=False,
            random_state=self.seed
        )

        idx_train_without_oracle, idx_test_without_oracle = train_test_split(
            indices_without,
            test_size=self.test_size_ratio,
            shuffle=False,
            random_state=self.seed
        )

        def get_shuffle_dataset(indices_with, indices_without):
            return concatenate_datasets([
                dataset_with_oracle.select(indices_with),
                dataset_without_oracle.select(indices_without)
            ]).shuffle(seed=self.seed)


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map='auto'
        )


    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )


    def format_data(self):
        def format_train_example(example):
            messages = [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["cot_answer"]}
            ]
            formatted = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            return {'formatted_text': formatted}


        def format_inference_example(example):
            messages = [
                {"role": "user", "content": example["instruction"]}
            ]
            formatted = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            return {'formatted_text': formatted}

        self.formatted_train_dataset = self.train_dataset.map(format_train_example)
        self.formatted_test_dataset = self.test_dataset.map(format_inference_example)
        self.formatted_eval_dataset = self.test_dataset.map(format_train_example)


    def tokenize(self):
        def tokenize_function(examples):
            return self.tokenizer(
                examples['formatted_text'],
                truncation=True,
                truncation_side='left',
                max_length=1024,
                padding=False,
                return_tensors=None,
            )

        self.tokenized_train_dataset = self.formatted_train_dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=self.formatted_train_dataset.column_names,
        )

        self.tokenized_test_dataset = self.formatted_test_dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=self.formatted_test_dataset.column_names,
        )

        self.tokenized_eval_dataset = self.formatted_eval_dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=self.formatted_eval_dataset.column_names,
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model


    def prepare_training(self):
        self.training_args = TrainingArguments(
            output_dir="./qwen-raft-lora",
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_steps=100,
            save_total_limit=2,
            fp16=True,
            report_to="none",
            remove_unused_columns=False,
        )

        self.data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False,
        )

        self.trainer = Trainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.tokenized_train_dataset,
            data_collator=self.data_collator,
        )


    def train(self):
        self.trainer.train()

    def evaluate_perplexity(self):
        pass


    def evaluate_f1(self):
        pass


    def run(self):
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.format_data()
        self.tokenize()
        self.add_peft_if_exist()
        self.prepare_training()
        self.train()
        self.evaluate_perplexity()
        self.evaluate_f1()

In [ ]:
exp = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=10,
    test_size_ratio=0.2,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1,
)